<a href="https://colab.research.google.com/github/rasecfaria/DeepLearning-Aplicado/blob/main/class_bin_OBS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports do dataset

In [9]:
!pip install ucimlrepo
!pip install pandas

In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition = fetch_ucirepo(id=544)

# data (as pandas dataframes)
X = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.features
y = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.targets

# metadata
print(estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.metadata)

# variable information
print(estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.variables)


{'uci_id': 544, 'name': 'Estimation of Obesity Levels Based On Eating Habits and Physical Condition ', 'repository_url': 'https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition', 'data_url': 'https://archive.ics.uci.edu/static/public/544/data.csv', 'abstract': 'This dataset include data for the estimation of obesity levels in individuals from the countries of Mexico, Peru and Colombia, based on their eating habits and physical condition. ', 'area': 'Health and Medicine', 'tasks': ['Classification', 'Regression', 'Clustering'], 'characteristics': ['Multivariate'], 'num_instances': 2111, 'num_features': 16, 'feature_types': ['Integer'], 'demographics': ['Gender', 'Age'], 'target_col': ['NObeyesdad'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Tue Sep 10 2024', 'dataset_doi': '10.24432/C5H31Z', 'creators': [], 'intro_paper': {'ID': 358, 'type': 

Meu codigo

In [3]:
# Ver as primeiras linhas do target
print(y.head())

# Ver todas as categorias únicas do target
print(y["NObeyesdad"].unique())

# Contar quantos exemplos tem em cada categoria
print(y["NObeyesdad"].value_counts())


            NObeyesdad
0        Normal_Weight
1        Normal_Weight
2        Normal_Weight
3   Overweight_Level_I
4  Overweight_Level_II
['Normal_Weight' 'Overweight_Level_I' 'Overweight_Level_II'
 'Obesity_Type_I' 'Insufficient_Weight' 'Obesity_Type_II'
 'Obesity_Type_III']
NObeyesdad
Obesity_Type_I         351
Obesity_Type_III       324
Obesity_Type_II        297
Overweight_Level_I     290
Overweight_Level_II    290
Normal_Weight          287
Insufficient_Weight    272
Name: count, dtype: int64


0 -> nao obeso

1 -> obeso


In [5]:
nao_obesos = ["Insufficient_Weight", "Normal_Weight",
              "Overweight_Level_I", "Overweight_Level_II"]

y["Target"] = y["NObeyesdad"].apply(lambda x: 0 if x in nao_obesos else 1)

print(y.head())
print(y["Target"].value_counts())


            NObeyesdad  Target
0        Normal_Weight       0
1        Normal_Weight       0
2        Normal_Weight       0
3   Overweight_Level_I       0
4  Overweight_Level_II       0
Target
0    1139
1     972
Name: count, dtype: int64


/tmp/ipython-input-331373789.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y["Target"] = y["NObeyesdad"].apply(lambda x: 0 if x in nao_obesos else 1)


ver o data types, no caso sao object e flaot64

é preciso converter os object em dummies(one-hot encoding)

**O Problema**
Imagine que você tem uma coluna de dados sobre a cor dos carros:

Cor
Vermelho
Azul
Verde
Vermelho

Exportar para Sheets
Um computador não consegue fazer cálculos com a palavra "Azul". Se tentássemos apenas substituir "Azul" por 1, "Vermelho" por 2 e "Verde" por 3, o modelo entenderia que o "Verde" (3) é três vezes maior ou melhor que o "Azul" (1), o que não faz sentido, pois cor não tem ordem!

A Solução: As "Dummies"
O One-Hot Encoding resolve isso criando uma nova coluna binária (0 ou 1) para cada categoria única na coluna original.

A coluna original é "explodida" em múltiplas colunas:

Cor	Cor_Vermelho	Cor_Azul	Cor_Verde
Vermelho	1	0	0
Azul	0	1	0
Verde	0	0	1
Vermelho	1	0	0



In [6]:
print(X.dtypes)

Gender                             object
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight     object
FAVC                               object
FCVC                              float64
NCP                               float64
CAEC                               object
SMOKE                              object
CH2O                              float64
SCC                                object
FAF                               float64
TUE                               float64
CALC                               object
MTRANS                             object
dtype: object


**Dummies**

In [11]:
import pandas as pd

categorical_cols = X.select_dtypes(include=["object"]).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

Normalizar os dados

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # devolve numpy array já escalado

Treino e Teste

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y["Target"],
    test_size=0.2,  # 20% teste
    random_state=42,
    stratify=y["Target"]  # garante equilíbrio das classes
)


tamanho do dataset

In [14]:
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (2111, 23)
Shape de y: (2111, 2)


distribuicao das classes

proporçao de obesos vs nao obesos

In [15]:
print(y["Target"].value_counts(normalize=True))

Target
0    0.539555
1    0.460445
Name: proportion, dtype: float64


Estatisticas dos dados

In [18]:
print(X.describe())

               Age       Height       Weight         FCVC          NCP  \
count  2111.000000  2111.000000  2111.000000  2111.000000  2111.000000   
mean     24.312600     1.701677    86.586058     2.419043     2.685628   
std       6.345968     0.093305    26.191172     0.533927     0.778039   
min      14.000000     1.450000    39.000000     1.000000     1.000000   
25%      19.947192     1.630000    65.473343     2.000000     2.658738   
50%      22.777890     1.700499    83.000000     2.385502     3.000000   
75%      26.000000     1.768464   107.430682     3.000000     3.000000   
max      61.000000     1.980000   173.000000     3.000000     4.000000   

              CH2O          FAF          TUE  
count  2111.000000  2111.000000  2111.000000  
mean      2.008011     1.010298     0.657866  
std       0.612953     0.850592     0.608927  
min       1.000000     0.000000     0.000000  
25%       1.584812     0.124505     0.000000  
50%       2.000000     1.000000     0.625350  
75% 

oooooo

In [21]:
import numpy as np
import tensorflow as tf

# reprodutibilidade
np.random.seed(42)
tf.random.set_seed(42)

# converter para numpy arrays do tipo float32
X_train = np.asarray(X_train, dtype=np.float32)
X_test  = np.asarray(X_test, dtype=np.float32)

# o target (y) deve ser 1D com 0 e 1
y_train = np.asarray(y_train).astype(np.float32).ravel()
y_test  = np.asarray(y_test).astype(np.float32).ravel()

# confirma os shapes
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)
print("Valores únicos em y_train:", np.unique(y_train, return_counts=True))
print("Valores únicos em y_test:", np.unique(y_test, return_counts=True))


X_train: (1688, 23) X_test: (423, 23)
y_train: (1688,) y_test: (423,)
Valores únicos em y_train: (array([0., 1.], dtype=float32), array([911, 777]))
Valores únicos em y_test: (array([0., 1.], dtype=float32), array([228, 195]))


modelo

In [22]:
from tensorflow.keras import models, layers

model = models.Sequential([
    layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  # primeira camada oculta
    layers.Dense(8, activation='relu'),                                    # segunda camada oculta
    layers.Dense(1, activation='sigmoid')                                  # saída binária
])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529 (2.07 KB)

 Trainable params: 529 (2.07 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
model.compile(
    optimizer='rmsprop',        # optimizador que atualiza os pesos durante o treino
    loss='binary_crossentropy',  # função de perda para classificação binária
    metrics=['accuracy']         # métrica que vamos acompanhar
)

treinar o modelo

In [24]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,             # número de vezes que a rede verá todos os dados de treino
    batch_size=32,         # quantos exemplos a rede vê antes de atualizar os pesos
    validation_split=0.2,  # 20% do treino usado para validar a performance a cada época
    verbose=2              # imprime progresso durante o treino
)

Epoch 1/30
43/43 - 2s - 42ms/step - accuracy: 0.6319 - loss: 0.6266 - val_accuracy: 0.7426 - val_loss: 0.5399
Epoch 2/30
43/43 - 0s - 4ms/step - accuracy: 0.7874 - loss: 0.4860 - val_accuracy: 0.8491 - val_loss: 0.4226
Epoch 3/30
43/43 - 0s - 4ms/step - accuracy: 0.8452 - loss: 0.3949 - val_accuracy: 0.8935 - val_loss: 0.3497
Epoch 4/30
43/43 - 0s - 4ms/step - accuracy: 0.8881 - loss: 0.3314 - val_accuracy: 0.9142 - val_loss: 0.2990
Epoch 5/30
43/43 - 0s - 4ms/step - accuracy: 0.9052 - loss: 0.2838 - val_accuracy: 0.9231 - val_loss: 0.2623
Epoch 6/30
43/43 - 0s - 5ms/step - accuracy: 0.9185 - loss: 0.2458 - val_accuracy: 0.9260 - val_loss: 0.2335
Epoch 7/30
43/43 - 0s - 4ms/step - accuracy: 0.9319 - loss: 0.2143 - val_accuracy: 0.9290 - val_loss: 0.2108
Epoch 8/30
43/43 - 0s - 4ms/step - accuracy: 0.9385 - loss: 0.1876 - val_accuracy: 0.9438 - val_loss: 0.1913
Epoch 9/30
43/43 - 0s - 4ms/step - accuracy: 0.9504 - loss: 0.1638 - val_accuracy: 0.9467 - val_loss: 0.1742
Epoch 10/30
43/43 

teste

In [25]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}  |  Test accuracy: {test_acc:.4f}")

Test loss: 0.0523  |  Test accuracy: 0.9787
